# 02_Diffraction_Sorting_SSA_v7
move away from previous methods to develop a human visual tiage approach
aims for three-stage pipeline with assisted learning in the future

In [1]:
# Cell 1: Imports and Set Up

#standard libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# for HDF/NXS file reading 
import h5py

import pyFAI




ModuleNotFoundError: No module named 'tqdm'

In [ ]:
# Cell 2: User inputs 
data_folder = "D:/I11 Beamtime July 25 - LHIST/sorting_v2_test"

# Create output folders if they don't exist
os.makedirs('02_V7_output', exist_ok=True)
os.makedirs('02_V7_output/processed', exist_ok=True)

print(f"Raw data folder: {data_folder}")


In [ ]:
# Cell 3 : Pixium Loader - Data Ingestion 
def load_pixium_frames(folder_path):
    frames = []
    file_paths = sorted([os.path.join(folder_path, f) for f in os.listdir(folder_path) 
                         if f.endswith(('.hdf', '.nxs'))])
    for fp in file_paths:
        try:
            with h5py.File(fp, 'r') as f:
                data = f['/entry/data/data'][:]  # adjust dataset path if needed
                frames.append({'file': fp, 'data': data})
        except Exception as e:
            print(f"Error loading {fp}: {e}")
    return frames

frames = load_pixium_frames(data_folder)
print(f"Loaded {len(frames)} frames from {data_folder}")


In [ ]:
# Cell 4 : Visual Triage 
def visual_triage(frames):
    results = []
    for i, frame in enumerate(frames):
        plt.imshow(np.log1p(frame['data']), cmap='viridis')
        plt.title(f"Frame {i+1}/{len(frames)}: {os.path.basename(frame['file'])}")
        plt.axis('off')
        plt.show(block=False)

        choice = input("y = diffraction, n = background, m = ambiguous: ").lower()
        while choice not in ['y', 'n', 'm']:
            choice = input("Invalid, choose y/n/m: ").lower()

        results.append({'file': frame['file'], 'label': choice})
        plt.close()
    
    df = pd.DataFrame(results)
    df.to_csv('output/classification_log.csv', index=False)
    return df

labels_df = visual_triage(frames)


In [ ]:
# Cell 5 : OPTIONAL - Organise File Based on Labels

# Create folders if they don't exist
for label in ['diffraction', 'background', 'maybe']:
    os.makedirs(f'output/{label}', exist_ok=True)

# Copy frames into their respective folders
import shutil

for idx, row in labels_df.iterrows():
    src = row['file']
    if row['label'] == 'y':
        dst = os.path.join('output/diffraction', os.path.basename(src))
    elif row['label'] == 'n':
        dst = os.path.join('output/background', os.path.basename(src))
    else:
        dst = os.path.join('output/maybe', os.path.basename(src))
    
    shutil.copy2(src, dst)  # copy2 preserves metadata


In [ ]:
# Cell 6: quantitative Analysis 
# Tells you what the diffraction actually looks like numerically. This is machine-precise — faster, reproducible

def process_diffraction(frame):
    """
    Placeholder for azimuthal integration + FEP subtraction
    """
    data = frame['data']
    I_q = np.sum(data, axis=0)   # temporary simple integration
    peaks = np.max(I_q)
    return I_q, peaks

diffraction_frames = [f for f in frames if labels_df.loc[labels_df['file'] == f['file'], 'label'].values[0] == 'y']

processed = []
for frame in tqdm(diffraction_frames):
    I_q, peaks = process_diffraction(frame)
    np.save(f'output/processed/{os.path.basename(frame["file"])}_Iq.npy', I_q)
    processed.append({'file': frame['file'], 'peaks': peaks})
    
processed_df = pd.DataFrame(processed)
processed_df.to_csv('output/processed/peaks_summary.csv', index=False)


In [ ]:
# Cell 7: OPTIONAL - Ambiguous Review
# combination of human and numeric info - mainly helpful for maybe frames
ambiguous_frames = [f for f in frames if labels_df.loc[labels_df['file'] == f['file'], 'label'].values[0] == 'm']

if ambiguous_frames:
    print(f"{len(ambiguous_frames)} ambiguous frames to review")
    labels_df_amb = visual_triage(ambiguous_frames)
    # Update the main labels dataframe
    labels_df.update(labels_df_amb)
    labels_df.to_csv('output/classification_log.csv', index=False)


In [ ]:
# Cell 8: Summary and Plotting
for idx, row in processed_df.iterrows():
    I_q = np.load(f'output/processed/{os.path.basename(row["file"])}_Iq.npy')
    plt.plot(I_q, label=os.path.basename(row['file']))
plt.xlabel('q (arb. units)')
plt.ylabel('Intensity (arb. units)')
plt.legend()
plt.show()
